# Main experiment

## Thư viện và CUDA xử lý GPU

In [ ]:
# Cell 1 — Kaggle runtime, thư viện và CUDA. Không tự cài dependency.
from pathlib import Path
import gc
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (
    AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, precision_score, recall_score,
    f1_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE

try:
    import torch
except ImportError:
    torch = None
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None
try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
except ImportError:
    TabNetClassifier = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_INPUT_ROOT.is_dir() and KAGGLE_WORKING_ROOT.is_dir()
if IS_KAGGLE:
    PROJECT_ROOT = KAGGLE_WORKING_ROOT / "FAIR_2026_Experiment"
else:
    local_cwd = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (p for p in (local_cwd, local_cwd.parent) if (p / "Notebook").is_dir()),
        local_cwd,
    )
RESULTS_ROOT = PROJECT_ROOT / "results" / "one_fold"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
SEED = RANDOM_STATE
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
CUDA_AVAILABLE = bool(torch is not None and torch.cuda.is_available())
GPU_COUNT = torch.cuda.device_count() if CUDA_AVAILABLE else 0
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)] if CUDA_AVAILABLE else []
if torch is not None:
    torch.manual_seed(RANDOM_STATE)
    if CUDA_AVAILABLE:
        torch.cuda.manual_seed_all(RANDOM_STATE)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

DATA_HINTS = {
    "MLG_ULB": {"creditcard.csv": KAGGLE_INPUT_ROOT / "mlg-ulb-creditcardfraud" / "creditcard.csv"},
    "IEEE_CIS": {
        "train_transaction.csv": KAGGLE_INPUT_ROOT / "ieee-fraud-detection" / "train_transaction.csv",
        "train_identity.csv": KAGGLE_INPUT_ROOT / "ieee-fraud-detection" / "train_identity.csv",
    },
    "SPARKOV": {
        "fraudTrain.csv": KAGGLE_INPUT_ROOT / "fraud-detection" / "fraudTrain.csv",
        "fraudTest.csv": KAGGLE_INPUT_ROOT / "fraud-detection" / "fraudTest.csv",
    },
}

def resolve_input_file(filename, hinted_path):
    if hinted_path.is_file():
        return hinted_path
    matches = list(KAGGLE_INPUT_ROOT.rglob(filename)) if IS_KAGGLE else list(PROJECT_ROOT.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(f"Cần đúng 1 file {filename}; tìm thấy {len(matches)}: {matches}")
    return matches[0]

PACKAGE_STATUS = {
    "xgboost": XGBClassifier is not None, "lightgbm": LGBMClassifier is not None,
    "catboost": CatBoostClassifier is not None, "pytorch_tabnet": TabNetClassifier is not None,
}
print(f"Kaggle={IS_KAGGLE} | CUDA={CUDA_AVAILABLE} | GPUs={GPU_NAMES}")
print("Package status:", PACKAGE_STATUS)
print("Results root:", RESULTS_ROOT)

## Trích xuất xử lý dữ liệu

Giới hạn: Chỉ lưu 1 fold để thực hiện huấn luyện ngay và lấy kết quả chạy với 13 models, sau đó giải phóng fold để chạy tiếp các fold còn lại.

Chọn dataset để thực hiện chạy

In [ ]:
# Cell 2 — chọn đúng một dataset cho mỗi lần chạy notebook.
DATASET_NAME = "MLG_ULB"  # MLG_ULB | IEEE_CIS | SPARKOV
if DATASET_NAME not in DATA_HINTS:
    raise ValueError(f"DATASET_NAME không hợp lệ: {DATASET_NAME}")
RESOLVED_INPUTS = {
    name: resolve_input_file(name, hint)
    for name, hint in DATA_HINTS[DATASET_NAME].items()
}
for name, path in RESOLVED_INPUTS.items():
    print(f"{name}: {path}")

Chọn fold thực hiện (từ 1 đến 5 ứng với mỗi loại dữ liệu thực hiện ULB, IEEE Fraud, Sparkov)

In [ ]:
# Cell 3 — cấu hình một fold và chế độ chạy nhanh.
SELECTED_FOLD = 1
N_SPLITS = 5
FAST_SMOKE = True
FAST_MAX_TRAIN_ROWS = 20_000
FAST_MAX_VALID_ROWS = 2_000
FAST_OHE_MAX_CATEGORIES = 32
REMOVE_MLG_DUPLICATES = False

if SELECTED_FOLD not in range(1, N_SPLITS + 1):
    raise ValueError(f"SELECTED_FOLD phải từ 1 đến {N_SPLITS}")
RUN_MODE = "fast_smoke" if FAST_SMOKE else "full_reproduction"
print({
    "dataset": DATASET_NAME, "fold": SELECTED_FOLD, "run_mode": RUN_MODE,
    "max_train_rows": FAST_MAX_TRAIN_ROWS if FAST_SMOKE else None,
    "max_valid_rows": FAST_MAX_VALID_ROWS if FAST_SMOKE else None,
})
if FAST_SMOKE:
    print("FAST_SMOKE dùng để kiểm tra end-to-end; metric không dùng làm kết quả bài báo.")

## Kiểm tra kết quả trích xuất và quan sát fold

In [ ]:
# Cell 4 — load, split, preprocessing train-only, SMOTE và validation gate.
def stratified_cap_indices(y, max_rows, seed):
    if max_rows is None or len(y) <= max_rows:
        return np.arange(len(y))
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=max_rows, random_state=seed)
    selected, _ = next(splitter.split(np.zeros(len(y)), y))
    return np.sort(selected)

def make_dense_ohe():
    kwargs = {"dtype": np.float32, "sparse_output": False}
    if FAST_SMOKE:
        kwargs.update(handle_unknown="infrequent_if_exist", max_categories=FAST_OHE_MAX_CATEGORIES)
    else:
        kwargs.update(handle_unknown="ignore")
    return OneHotEncoder(**kwargs)

# 1) Load và xử lý deterministic theo từng dataset.
id_series = None
preprocessing_notes = []
if DATASET_NAME == "MLG_ULB":
    raw_df = pd.read_csv(RESOLVED_INPUTS["creditcard.csv"])
    expected = ["Time", *[f"V{i}" for i in range(1, 29)], "Amount", "Class"]
    missing_required = [c for c in expected if c not in raw_df.columns]
    assert not missing_required, f"Thiếu cột MLG: {missing_required}"
    duplicate_count = int(raw_df.duplicated().sum())
    if REMOVE_MLG_DUPLICATES:
        raw_df = raw_df.drop_duplicates().reset_index(drop=True)
    X_raw = raw_df.drop(columns=["Class"]).copy()
    y_raw = raw_df["Class"].astype(np.int8).copy()
    target_name = "Class"
    preprocessing_notes.append(f"duplicates={duplicate_count}, removed={REMOVE_MLG_DUPLICATES}")
elif DATASET_NAME == "IEEE_CIS":
    tx = pd.read_csv(RESOLVED_INPUTS["train_transaction.csv"])
    identity = pd.read_csv(RESOLVED_INPUTS["train_identity.csv"])
    raw_df = tx.merge(identity, on="TransactionID", how="left")
    assert len(raw_df) == len(tx), "LEFT JOIN IEEE làm thay đổi số transaction"
    id_series = raw_df["TransactionID"].copy()
    y_raw = raw_df["isFraud"].astype(np.int8).copy()
    X_raw = raw_df.drop(columns=["isFraud", "TransactionID"]).copy()
    target_name = "isFraud"
    preprocessing_notes.append("IEEE transaction LEFT JOIN identity; TransactionID excluded")
    del tx, identity
else:
    raw_df = pd.read_csv(RESOLVED_INPUTS["fraudTrain.csv"])
    if "Unnamed: 0" in raw_df.columns:
        raw_df = raw_df.drop(columns=["Unnamed: 0"])
    id_series = raw_df["trans_num"].copy() if "trans_num" in raw_df.columns else None
    raw_df = raw_df.drop(columns=[c for c in ["trans_num", "cc_num", "first", "last", "street"] if c in raw_df.columns])
    tx_dt = pd.to_datetime(raw_df["trans_date_trans_time"], errors="coerce")
    raw_df["transaction_hour"] = tx_dt.dt.hour
    raw_df["transaction_day"] = tx_dt.dt.day
    raw_df["transaction_month"] = tx_dt.dt.month
    raw_df["transaction_weekday"] = tx_dt.dt.dayofweek
    raw_df["is_weekend"] = (tx_dt.dt.dayofweek >= 5).astype(np.int8)
    raw_df = raw_df.drop(columns=["trans_date_trans_time"])
    y_raw = raw_df["is_fraud"].astype(np.int8).copy()
    X_raw = raw_df.drop(columns=["is_fraud"]).copy()
    target_name = "is_fraud"
    preprocessing_notes.append("Sparkov time features enabled; technical/person IDs excluded")

# 2) Tạo fold trên toàn bộ dữ liệu trước mọi fit preprocessing.
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
all_fold_indices = list(skf.split(X_raw, y_raw))
full_train_idx, full_valid_idx = all_fold_indices[SELECTED_FOLD - 1]
assert len(np.intersect1d(full_train_idx, full_valid_idx)) == 0

# IEEE: danh sách drop missing được học từ full training fold, không từ validation.
drop_cols = []
if DATASET_NAME == "IEEE_CIS":
    train_missing_ratio = X_raw.iloc[full_train_idx].isna().mean()
    drop_cols = train_missing_ratio[train_missing_ratio > 0.50].index.tolist()
    X_raw = X_raw.drop(columns=drop_cols)
    preprocessing_notes.append(f"drop_missing_gt_50pct_from_train={len(drop_cols)}")

# 3) FAST_SMOKE chỉ cap sau khi fold thật đã được khóa.
train_local = stratified_cap_indices(y_raw.iloc[full_train_idx].reset_index(drop=True), FAST_MAX_TRAIN_ROWS if FAST_SMOKE else None, RANDOM_STATE)
valid_local = stratified_cap_indices(y_raw.iloc[full_valid_idx].reset_index(drop=True), FAST_MAX_VALID_ROWS if FAST_SMOKE else None, RANDOM_STATE + 1)
train_source_idx = full_train_idx[train_local]
valid_source_idx = full_valid_idx[valid_local]
X_train_raw = X_raw.iloc[train_source_idx].copy()
y_train_raw = y_raw.iloc[train_source_idx].copy()
X_valid_raw = X_raw.iloc[valid_source_idx].copy()
y_valid = y_raw.iloc[valid_source_idx].to_numpy(dtype=np.int8)

# 4) Fit preprocessing trên train và transform validation.
if DATASET_NAME == "MLG_ULB":
    preprocessor = StandardScaler()
    X_train_proc = preprocessor.fit_transform(X_train_raw).astype(np.float32)
    X_valid = preprocessor.transform(X_valid_raw).astype(np.float32)
    feature_names = X_train_raw.columns.astype(str).tolist()
else:
    categorical_cols = X_train_raw.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numeric_cols = [c for c in X_train_raw.columns if c not in categorical_cols]
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=-999)),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", make_dense_ohe()),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ])
    X_train_proc = preprocessor.fit_transform(X_train_raw).astype(np.float32)
    X_valid = preprocessor.transform(X_valid_raw).astype(np.float32)
    feature_names = preprocessor.get_feature_names_out().astype(str).tolist()
    if FAST_SMOKE:
        preprocessing_notes.append(f"max_categories={FAST_OHE_MAX_CATEGORIES} (smoke adaptation)")

minority_count = int(y_train_raw.value_counts().min())
assert minority_count > 5, f"Không đủ fraud cho SMOTE k=5: {minority_count}"
smote = SMOTE(sampling_strategy=1.0, k_neighbors=5, random_state=RANDOM_STATE)
X_train, y_train = smote.fit_resample(X_train_proc, y_train_raw)
X_train = np.asarray(X_train, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.int8)

# 5) Validation gate bắt buộc trước model.
checks = {
    "source_indices_disjoint": len(np.intersect1d(train_source_idx, valid_source_idx)) == 0,
    "feature_count_matches": X_train.shape[1] == X_valid.shape[1] == len(feature_names),
    "train_no_nan": not np.isnan(X_train).any(),
    "valid_no_nan": not np.isnan(X_valid).any(),
    "train_no_inf": np.isfinite(X_train).all(),
    "valid_no_inf": np.isfinite(X_valid).all(),
    "smote_train_balanced": len(np.unique(np.bincount(y_train))) == 1,
    "validation_not_smoted": len(y_valid) == len(valid_source_idx),
    "both_classes_present": set(np.unique(y_valid)) == {0, 1},
}
PREPROCESSING_VALIDATED = all(checks.values())
if not PREPROCESSING_VALIDATED:
    raise RuntimeError(f"Preprocessing validation failed: {checks}")
valid_reference = pd.DataFrame({"source_index": valid_source_idx, target_name: y_valid})
if id_series is not None:
    valid_reference["record_id"] = id_series.iloc[valid_source_idx].to_numpy()
audit = pd.DataFrame([{
    "dataset": DATASET_NAME, "fold": SELECTED_FOLD, "run_mode": RUN_MODE,
    "raw_rows": len(y_raw), "train_before_smote": len(y_train_raw),
    "train_after_smote": len(y_train), "validation_rows": len(y_valid),
    "features": X_train.shape[1], "validation_fraud_rate": float(y_valid.mean()),
    "train_matrix_gb": X_train.nbytes / 1024**3,
    "notes": "; ".join(preprocessing_notes),
}])
DATA_VARIANT = ("without_duplicates" if REMOVE_MLG_DUPLICATES else "with_duplicates") if DATASET_NAME == "MLG_ULB" else "reproduction"
RESULT_DATASET_ROOT = RESULTS_ROOT / DATASET_NAME / DATA_VARIANT
run_dir = RESULT_DATASET_ROOT / f"fold_{SELECTED_FOLD:02d}" / RUN_MODE
run_dir.mkdir(parents=True, exist_ok=True)
audit.to_csv(run_dir / "preprocessing_audit.csv", index=False)
valid_reference.to_csv(run_dir / "validation_reference.csv", index=False)
display(pd.DataFrame({"check": checks.keys(), "passed": checks.values()}))
display(audit)
print("Train class:", np.bincount(y_train), "Validation class:", np.bincount(y_valid))
print("PREPROCESSING_VALIDATED =", PREPROCESSING_VALIDATED)
del raw_df, X_raw, y_raw, X_train_raw, X_valid_raw, X_train_proc
gc.collect()

## Trích xuất mô hình (13 mô hình chính)

In [ ]:
# Cell 5 — cổng kiểm định và hàm đánh giá dùng chung cho cả 13 mô hình.
if not globals().get("PREPROCESSING_VALIDATED", False):
    raise RuntimeError("Dữ liệu chưa vượt qua kiểm định ở Cell 4; không được huấn luyện.")

experiment_rows = []

def record_unavailable(model_name, reason):
    row = {
        "dataset": DATASET_NAME, "variant": DATA_VARIANT, "fold": SELECTED_FOLD, "run_mode": RUN_MODE,
        "model": model_name, "status": "unavailable", "device": "-",
        "precision": np.nan, "recall": np.nan, "f1": np.nan,
        "roc_auc": np.nan, "pr_auc": np.nan,
        "tn": np.nan, "fp": np.nan, "fn": np.nan, "tp": np.nan,
        "fit_seconds": np.nan, "train_rows": len(y_train),
        "valid_rows": len(y_valid), "n_features": X_train.shape[1],
        "note": str(reason)[:500],
    }
    experiment_rows.append(row)
    print(f"{model_name}: unavailable — {reason}")

def evaluate_fitted_model(model_name, model, fit_seconds, device, note=""):
    y_pred = np.asarray(model.predict(X_valid)).reshape(-1).astype(np.int8)
    if hasattr(model, "predict_proba"):
        y_score = np.asarray(model.predict_proba(X_valid))[:, 1]
    elif hasattr(model, "decision_function"):
        raw_score = np.asarray(model.decision_function(X_valid)).reshape(-1)
        y_score = 1.0 / (1.0 + np.exp(-np.clip(raw_score, -30, 30)))
    else:
        y_score = y_pred.astype(np.float32)
    tn, fp, fn, tp = confusion_matrix(y_valid, y_pred, labels=[0, 1]).ravel()
    row = {
        "dataset": DATASET_NAME, "variant": DATA_VARIANT, "fold": SELECTED_FOLD, "run_mode": RUN_MODE,
        "model": model_name, "status": "ok", "device": device,
        "precision": precision_score(y_valid, y_pred, zero_division=0),
        "recall": recall_score(y_valid, y_pred, zero_division=0),
        "f1": f1_score(y_valid, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_valid, y_score),
        "pr_auc": average_precision_score(y_valid, y_score),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "fit_seconds": float(fit_seconds), "train_rows": len(y_train),
        "valid_rows": len(y_valid), "n_features": X_train.shape[1], "note": note,
    }
    experiment_rows.append(row)
    print(f"{model_name}: F1={row['f1']:.6f}, PR-AUC={row['pr_auc']:.6f}, {fit_seconds:.1f}s, {device}")

def fit_and_evaluate(model_name, estimator, device="CPU", fallback=None):
    start = time.perf_counter()
    used_device, note = device, ""
    try:
        estimator.fit(X_train, y_train)
    except Exception as exc:
        if fallback is None:
            record_unavailable(model_name, exc)
            return
        note = f"GPU fallback: {type(exc).__name__}: {exc}"[:500]
        print(f"{model_name}: GPU không khả dụng, chuyển sang CPU.")
        del estimator
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
        estimator, used_device = fallback(), "CPU fallback"
        start = time.perf_counter()
        try:
            estimator.fit(X_train, y_train)
        except Exception as fallback_exc:
            record_unavailable(model_name, fallback_exc)
            return
    evaluate_fitted_model(model_name, estimator, time.perf_counter() - start, used_device, note)
    del estimator
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Cổng kiểm định đã mở. Bắt đầu huấn luyện tuần tự để giới hạn RAM/VRAM.")

### Khởi tạo toàn bộ mô hình nghiên cứu

### Huấn luyện và đánh giá mô hình

#### 1. Mô hình Logistic Regression

In [ ]:
fit_and_evaluate(
    "Logistic Regression",
    LogisticRegression(max_iter=300, solver="liblinear", random_state=SEED),
)

#### 2. Mô hình Decision Tree

In [ ]:
fit_and_evaluate(
    "Decision Tree",
    DecisionTreeClassifier(random_state=SEED),
)

#### 3. Mô hình Random Forest

In [ ]:
fit_and_evaluate(
    "Random Forest",
    RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
)

#### 4. Mô hình LightGBM

In [ ]:
if LGBMClassifier is None:
    record_unavailable("LightGBM", "lightgbm chưa có trong Kaggle image")
else:
    lgbm_common = dict(n_estimators=200, learning_rate=0.1, objective="binary", random_state=SEED, verbosity=-1)
    lgbm_gpu = LGBMClassifier(**lgbm_common, device_type="gpu") if CUDA_AVAILABLE else LGBMClassifier(**lgbm_common)
    lgbm_fallback = (lambda: LGBMClassifier(**lgbm_common)) if CUDA_AVAILABLE else None
    fit_and_evaluate("LightGBM", lgbm_gpu, "GPU" if CUDA_AVAILABLE else "CPU", lgbm_fallback)

#### 5. Mô hình CatBoost

In [ ]:
if CatBoostClassifier is None:
    record_unavailable("CatBoost", "catboost chưa có trong Kaggle image")
else:
    cat_common = dict(iterations=200, learning_rate=0.1, loss_function="Logloss", verbose=False, random_seed=SEED)
    cat_devices = ":".join(str(i) for i in range(GPU_COUNT))
    cat_gpu = CatBoostClassifier(**cat_common, task_type="GPU", devices=cat_devices) if CUDA_AVAILABLE else CatBoostClassifier(**cat_common)
    cat_fallback = (lambda: CatBoostClassifier(**cat_common)) if CUDA_AVAILABLE else None
    fit_and_evaluate("CatBoost", cat_gpu, "GPU" if CUDA_AVAILABLE else "CPU", cat_fallback)

#### 6. Mô hình XGBoost

In [ ]:
if XGBClassifier is None:
    record_unavailable("XGBoost", "xgboost chưa có trong Kaggle image")
else:
    xgb_common = dict(n_estimators=200, learning_rate=0.1, max_depth=6, tree_method="hist", eval_metric="logloss", random_state=SEED, n_jobs=-1)
    xgb_gpu = XGBClassifier(**xgb_common, device="cuda") if CUDA_AVAILABLE else XGBClassifier(**xgb_common)
    xgb_fallback = (lambda: XGBClassifier(**xgb_common, device="cpu")) if CUDA_AVAILABLE else None
    fit_and_evaluate("XGBoost", xgb_gpu, "GPU" if CUDA_AVAILABLE else "CPU", xgb_fallback)

#### 7. Mô hình AdaBoost

In [ ]:
fit_and_evaluate(
    "AdaBoost",
    AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED),
)

#### 8. Mô hình TabNet

In [ ]:
if TabNetClassifier is None or torch is None:
    record_unavailable("TabNet", "pytorch-tabnet hoặc torch chưa có trong Kaggle image")
else:
    tabnet = TabNetClassifier(
        n_d=32, n_a=32, n_steps=5, gamma=1.5, seed=SEED,
        optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=0.02),
        device_name="cuda" if CUDA_AVAILABLE else "cpu", verbose=10,
    )
    start = time.perf_counter()
    try:
        tabnet.fit(
            X_train, y_train, eval_set=[(X_valid, y_valid)], eval_name=["valid"],
            eval_metric=["auc"], max_epochs=20 if FAST_SMOKE else 100,
            patience=5 if FAST_SMOKE else 15, batch_size=1024,
            virtual_batch_size=128, num_workers=0, drop_last=False,
        )
        evaluate_fitted_model("TabNet", tabnet, time.perf_counter() - start, "GPU" if CUDA_AVAILABLE else "CPU")
    except Exception as exc:
        record_unavailable("TabNet", exc)
    del tabnet
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()

#### 9. Mô hình Extra Trees

In [ ]:
fit_and_evaluate(
    "Extra Trees",
    ExtraTreesClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
)

#### 10. Mô hình KNN

In [ ]:
fit_and_evaluate(
    "KNN",
    KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
)

#### 11. Mô hình LDA

In [ ]:
fit_and_evaluate(
    "LDA",
    LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
)

#### 12. Mô hình Naive Bayes

In [ ]:
fit_and_evaluate(
    "Naive Bayes",
    GaussianNB(),
)

#### 13. Mô hình Gradient Boosting

In [ ]:
fit_and_evaluate(
    "Gradient Boosting",
    GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED),
)

### Ma trận và các chỉ số đánh giá theo nghiên cứu

In [ ]:
# Cell 19 — bảng metrics và confusion matrix của toàn bộ mô hình đã chạy.
results_df = pd.DataFrame(experiment_rows)
expected_models = [
    "Logistic Regression", "Decision Tree", "Random Forest", "LightGBM",
    "CatBoost", "XGBoost", "AdaBoost", "TabNet", "Extra Trees",
    "KNN", "LDA", "Naive Bayes", "Gradient Boosting",
]
assert results_df["model"].tolist() == expected_models, "Phải chạy tuần tự đủ 13 cell mô hình"
metric_cols = ["precision", "recall", "f1", "roc_auc", "pr_auc"]
display(results_df[["model", "status", "device", *metric_cols, "fit_seconds"]].sort_values("f1", ascending=False))

ok_df = results_df.query("status == 'ok'").copy()
if not ok_df.empty:
    fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(ok_df))))
    plot_df = ok_df.sort_values("f1")
    ax.barh(plot_df["model"], plot_df["f1"], color="#276FBF")
    ax.set(xlabel="F1", title=f"{DATASET_NAME} — fold {SELECTED_FOLD} — {RUN_MODE}", xlim=(0, 1))
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

### Lưu kết quả fold tương ứng với dataset tương ứng

Lưu vào dataset tương ứng với fold tương ứng. Sau đó sang cell tiếp theo sẽ khởi chạy kết quả đánh giá hiệu suất các mô hình trên tập dataset tương ứng một khi đủ toàn bộ các fold (chỉ chạy khi đủ toàn bộ fold)

In [ ]:
# Cell 20 — chỉ lưu kết quả nhỏ; không lưu ma trận train/validation hay model.
metrics_path = run_dir / "model_metrics.csv"
features_path = run_dir / "feature_names.csv"
results_df.to_csv(metrics_path, index=False)
pd.DataFrame({"feature": feature_names}).to_csv(features_path, index=False)
print(f"Đã lưu: {metrics_path}")
print(f"Kích thước thư mục kết quả: {sum(p.stat().st_size for p in run_dir.glob('*') if p.is_file()) / 1024**2:.2f} MiB")

Lưu kết quả của dataset với 13 mô hình các mectrics với trung bình và cả độ lệch chuẩn.

In [ ]:
# Cell 21 — tổng hợp mean/std khi cùng dataset và run_mode đã có đủ 5 fold.
fold_metric_paths = sorted(RESULT_DATASET_ROOT.glob(f"fold_*/{RUN_MODE}/model_metrics.csv"))
available_folds = sorted({int(path.parents[1].name.split('_')[-1]) for path in fold_metric_paths})
print("Fold đã có:", available_folds)
if available_folds != list(range(1, N_SPLITS + 1)):
    print(f"Chưa tổng hợp: cần đủ fold 1..{N_SPLITS} cho {DATASET_NAME}/{RUN_MODE}.")
else:
    all_folds_df = pd.concat([pd.read_csv(path) for path in fold_metric_paths], ignore_index=True)
    successful = all_folds_df.query("status == 'ok'")
    summary_df = successful.groupby("model")[["precision", "recall", "f1", "roc_auc", "pr_auc"]].agg(["mean", "std"])
    summary_df.columns = [f"{metric}_{stat}" for metric, stat in summary_df.columns]
    summary_df = summary_df.reset_index().sort_values("f1_mean", ascending=False)
    summary_path = RESULT_DATASET_ROOT / f"summary_{RUN_MODE}_5fold.csv"
    summary_df.to_csv(summary_path, index=False)
    display(summary_df)
    print(f"Đã lưu tổng hợp 5-fold: {summary_path}")

# Giải phóng ma trận lớn sau khi đã lưu metrics.
del X_train, X_valid, y_train, y_valid
gc.collect()
if torch is not None and torch.cuda.is_available():
    torch.cuda.empty_cache()